# <span style='color:blue'> LAB 8: </span>
# <span style='color:blue'> GENERATIVE ADVERSARIAL NETWORKS </span>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns

In [ ]:
# Seaborn plot styling
sns.set(style = 'white', font_scale = 2)

# MNIST Generation with DCGAN

## Prepare Data

In [ ]:
from torchvision.datasets import MNIST 
from torch.utils.data import DataLoader
from torchvision import transforms

# Define a transformation to convert the data into Tensors
train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Scale to [-1, 1] for Tanh generator output
])

# Download the train and test MNIST data and transform it into Tensors
train_data = MNIST(root="./train.", train=True, download=True, transform=train_transforms)

## Define Model

In [ ]:
class Generator(torch.nn.Module):
    
    def __init__(self, input_noise_dim, feature_maps=128):
        
        super(Generator, self).__init__()
        
        self.input_noise_dim = input_noise_dim  # Dimension of the latent vector z
        self.feature_maps = feature_maps  # Channel multiplier
        
        self.net = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(input_noise_dim, feature_maps * 4, kernel_size=7, stride=1, padding=0, bias=False),
            torch.nn.BatchNorm2d(feature_maps * 4),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps * 2),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(feature_maps * 2, feature_maps, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(feature_maps, 1, kernel_size=3, stride=1, padding=1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, noise):
        
        noise = noise.view(-1, self.input_noise_dim, 1, 1)
        out = self.net(noise)
        
        return out

class Discriminator(torch.nn.Module):
    
    def __init__(self, feature_maps=64):
        
        super(Discriminator, self).__init__()
        
        self.feature_maps = feature_maps  # Channel multiplier
        
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, feature_maps, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.LeakyReLU(0.2, inplace=True),
            torch.nn.Conv2d(feature_maps, feature_maps * 2, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps * 2),
            torch.nn.LeakyReLU(0.2, inplace=True),
            torch.nn.Conv2d(feature_maps * 2, feature_maps * 4, kernel_size=3, stride=1, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps * 4),
            torch.nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.dropout = torch.nn.Dropout2d(0.3)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(feature_maps * 4 * 7 * 7, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        
        features = self.dropout(self.features(x))
        flat = features.view(x.size(0), -1)
        out = self.classifier(flat)
        
        return out.view(-1, 1).squeeze(1)

## Define Hyperparameters

In [ ]:
# Fix random seed
torch.manual_seed(42)

# Define learning rate + epochs
learning_rate = 0.001
epochs = 10

batchsize = 128
input_noise_dim = 100
feature_maps = 64
disc_steps = 3

# Weight initialization borrowed from the DCGAN paper
def weights_init(module):
    classname = module.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        torch.nn.init.normal_(module.weight.data, 0.0, 0.02)
        if hasattr(module, 'bias') and module.bias is not None:
            torch.nn.init.constant_(module.bias.data, 0)
    elif classname.find('BatchNorm') != -1:
        torch.nn.init.normal_(module.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(module.bias.data, 0)

# Initialize models
disc = Discriminator(feature_maps)
gen = Generator(input_noise_dim, feature_maps * 2)

disc.apply(weights_init)
gen.apply(weights_init)

# Binary Cross Entropy (BCE) loss function
loss_func = torch.nn.BCELoss()
optimizer_disc = torch.optim.Adam(disc.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_gen = torch.optim.Adam(gen.parameters(), lr=learning_rate, betas=(0.5, 0.999))

# Determine the device for training (GPU if available, otherwise CPU)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
disc.to(device)
gen.to(device)

## Identify Tracked Values

In [ ]:
gen_train_loss_list = []
disc_train_loss_list = []

## Train Model

In [ ]:
# Create DataLoader objects to efficiently load the training data in batches
train_loader = DataLoader(train_data, batch_size=batchsize, shuffle=True, drop_last=True)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Reset tracked losses each time the training cell is executed
disc_train_loss_list = []
gen_train_loss_list = []

for epoch in range(epochs):
    
    print('Epoch {}/{}'.format(epoch + 1, epochs))
    running_loss_D = 0.0
    running_loss_G = 0.0
    
    for inputs, _ in train_loader:
        
        inputs = inputs.to(device)
        
        # Label smoothing 
        real_label = torch.full((batchsize,), 0.9, dtype=inputs.dtype, device=device)
        fake_label = torch.full((batchsize,), 0.1, dtype=inputs.dtype, device=device)

        # Train Discriminator desc_steps times
        for _ in range(disc_steps):
            optimizer_disc.zero_grad()

            output_real = disc(inputs)
            D_real_loss = loss_func(output_real, real_label)
            D_real_loss.backward()

            noise = torch.randn(batchsize, input_noise_dim, device=device)
            fake = gen(noise)
            output_fake = disc(fake.detach())
            D_fake_loss = loss_func(output_fake, fake_label)
            D_fake_loss.backward()

            Disc_loss = D_real_loss + D_fake_loss
            optimizer_disc.step()

            running_loss_D += Disc_loss.item()

        # Train Generator ----------------------------------------------------------------------------------------
        optimizer_gen.zero_grad()
        noise = torch.randn(batchsize, input_noise_dim, device=device)
        fake = gen(noise)
        output = disc(fake)

        Gen_loss = loss_func(output, real_label)
        Gen_loss.backward()
        optimizer_gen.step()

        running_loss_G += Gen_loss.item()

    avg_disc_loss = running_loss_D / (len(train_loader) * disc_steps)
    avg_gen_loss = running_loss_G / len(train_loader)

    disc_train_loss_list.append(avg_disc_loss)
    gen_train_loss_list.append(avg_gen_loss)

    # Print the losses for the current epoch
    print("Discriminator Loss : {}".format(avg_disc_loss))
    print("Generator Loss : {}".format(avg_gen_loss))

## Visualize & Evaluate Model

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
epoch_axis = range(1, len(disc_train_loss_list) + 1)

plt.plot(epoch_axis, disc_train_loss_list, label='Discriminator Loss')
plt.plot(epoch_axis, gen_train_loss_list, label='Generator Loss')

plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('DCGAN Training Loss per Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def show_image(img):
    # Convert the image from a tensor to a NumPy array
    npimg = img.numpy()
    # Denormalize back to [0, 1] for display
    npimg = (npimg + 1) / 2.0
    npimg = np.clip(npimg, 0, 1)
    # Transpose the NumPy array to the correct format for displaying
    npimg = np.transpose(npimg, (1, 2, 0))
    # Handle grayscale images (1 channel) - squeeze out the channel dimension
    if npimg.shape[2] == 1:
        npimg = npimg.squeeze(2)
        plt.imshow(npimg, cmap='gray')
    else:
        plt.imshow(npimg)

In [ ]:
import torchvision

# Random noise for generating fake images
random_noise = torch.randn(128, input_noise_dim, device=device)

with torch.no_grad():
    

    fake = gen(random_noise)
    fake = fake.cpu()  
    
    fig, ax = plt.subplots(figsize=(20, 8.5))
    
    show_image(torchvision.utils.make_grid(fake[0:50], 10, 5))
    
    plt.show()

# Notes

## Vanilla
- The initial descriminator network did a terrible job

- Loss tell me very little so each change was more sujbect than previous models

- Adding multiple discriminator updates before generator seemed to make things worse

- Data normalization did the most because it made it stable for longer trainings

- Added Label smoothing instead of 1,0. Seemed to have some positive impact

## DCGAN

- Output much, much cleaner

- Setup was much more involved and had much more hard coded values. Modifying all of them seem like a headache

- Updating descriminator steps made much more of a difference in DCGAN

- Number of epochs needed dropped dramatically, even accounting for multiple descriminator itterations per loop


